# AgrowAI Chilli V2 training\nProvisional public-dataset baseline only. Farmer access remains disabled.

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\nPROJECT_ROOT='/content/drive/MyDrive/AgrowAI_V1/chilli_retraining_v2'\nPACKAGE_ROOT=PROJECT_ROOT\nDATA_ROOT=PROJECT_ROOT\nOUTPUT_ROOT=PROJECT_ROOT+'/outputs'\nprint('PACKAGE_VERSION=1.0.0')

In [ ]:
import torch\nassert torch.cuda.is_available(), 'CUDA_GPU_REQUIRED'\nname=torch.cuda.get_device_name(0)\nprint(name)\nassert any(x in name.upper() for x in ['T4','L4','A100','V100']), 'T4_OR_BETTER_REQUIRED'

In [ ]:
!pip install -q -r {PACKAGE_ROOT}/requirements.txt

In [ ]:
!python {PACKAGE_ROOT}/verify_training_inputs.py --package-root {PACKAGE_ROOT} --data-root {DATA_ROOT} --verify-images

In [ ]:
!python {PACKAGE_ROOT}/train.py --package-root {PACKAGE_ROOT} --data-root {DATA_ROOT} --output-dir {OUTPUT_ROOT}/training

In [ ]:
!python {PACKAGE_ROOT}/evaluate.py --package-root {PACKAGE_ROOT} --data-root {DATA_ROOT} --checkpoint {OUTPUT_ROOT}/training/best.pt --split development_validation --output-dir {OUTPUT_ROOT}/development\n!python {PACKAGE_ROOT}/evaluate.py --package-root {PACKAGE_ROOT} --data-root {DATA_ROOT} --checkpoint {OUTPUT_ROOT}/training/best.pt --split calibration --output-dir {OUTPUT_ROOT}/calibration

In [ ]:
!python {PACKAGE_ROOT}/calibration.py --logits {OUTPUT_ROOT}/calibration/calibration_logits.npz --output {OUTPUT_ROOT}/calibration/calibration.json\n!python {PACKAGE_ROOT}/abstention.py --logits {OUTPUT_ROOT}/calibration/calibration_logits.npz --calibration {OUTPUT_ROOT}/calibration/calibration.json --output {OUTPUT_ROOT}/calibration/thresholds.json

In [ ]:
ALLOW_ONE_TIME_PROVISIONAL_TEST=False\nassert ALLOW_ONE_TIME_PROVISIONAL_TEST is True, 'EXPLICIT_ONE_TIME_CONFIRMATION_REQUIRED'\n# Run only after model/calibration/threshold freeze. Never tune afterward.\n!python {PACKAGE_ROOT}/evaluate.py --package-root {PACKAGE_ROOT} --data-root {DATA_ROOT} --checkpoint {OUTPUT_ROOT}/training/best.pt --split provisional_test --allow-one-time-provisional-test --output-dir {OUTPUT_ROOT}/provisional_test

In [ ]:
!python {PACKAGE_ROOT}/package_runtime.py --package-root {PACKAGE_ROOT} --checkpoint {OUTPUT_ROOT}/training/best.pt --calibration {OUTPUT_ROOT}/calibration/calibration.json --thresholds {OUTPUT_ROOT}/calibration/thresholds.json --evaluation {OUTPUT_ROOT}/provisional_test/provisional_test_metrics.json --output-dir {OUTPUT_ROOT}/runtime\n!python {PACKAGE_ROOT}/verify_runtime.py {OUTPUT_ROOT}/runtime

In [ ]:
import hashlib, pathlib\ndef sha256(p):\n h=hashlib.sha256()\n with open(p,'rb') as f:\n  for b in iter(lambda:f.read(1048576),b''):h.update(b)\n return h.hexdigest()\nfor p in sorted(pathlib.Path(OUTPUT_ROOT).rglob('*')):\n if p.is_file(): print(sha256(p),p)\nprint('OUTPUTS_PERSISTED_TO_DRIVE')\nprint('FARMER_ACCESS_GATES_UNCHANGED')